Este es el notebook para el ejercicio propuesto en el TP Final de la materia Vision por Computadora.

In [ ]:
pip install opencv-python

1. Montar el drive de Google para leer los datos de la carpeta correspondiente.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


La siguiente celda permite elegir una carpeta para cargar las imágenes de las emociones. Para gestionar efectivamente la memoria se aplica el preprocesamiento convirtiendo a escala de grises y redimensionando de acuerdo a parámetros establecidos por el usuario


In [ ]:
import os
import cv2
import matplotlib.pyplot as plt

# Pedir al usuario la ruta de la carpeta principal que contiene las carpetas de emociones
image_base_folder_path = input("Por favor, introduce la ruta completa a la carpeta principal que contiene las carpetas de emociones (ej: /content/drive/MyDrive/00.DataVxC/emotions_dataset/): ")

# Solicitar al usuario las dimensiones deseadas para las imágenes pre-procesadas
while True:
    try:
        target_width = int(input("Introduce el ancho deseado para las imágenes pre-procesadas (ej: 48): "))
        target_height = int(input("Introduce el alto deseado para las imágenes pre-procesadas (ej: 48): "))
        if target_width > 0 and target_height > 0:
            break
        else:
            print("Las dimensiones deben ser números positivos. Intenta de nuevo.")
    except ValueError:
        print("Entrada inválida. Por favor, introduce un número entero.")

target_dim = (target_width, target_height)
print(f"Las imágenes se pre-procesarán a dimensiones: {target_dim} en escala de grises.")

# Verificar si la carpeta base existe
if not os.path.isdir(image_base_folder_path):
    print(f"Error: La carpeta '{image_base_folder_path}' no existe. Por favor, verifica la ruta.")
else:
    print(f"Procesando imágenes de las subcarpetas en: {image_base_folder_path}")
    processed_images = [] # Lista para almacenar las imágenes ya procesadas
    image_filenames = [] # Lista para almacenar los nombres de archivo relativos
    image_labels = [] # Lista para almacenar las etiquetas de las emociones

    # Extensiones de imagen comunes
    image_extensions = ('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff')

    # Iterar sobre los elementos dentro de la carpeta base
    for emotion_folder_name in os.listdir(image_base_folder_path):
        emotion_folder_path = os.path.join(image_base_folder_path, emotion_folder_name)

        # Verificar si es un directorio
        if os.path.isdir(emotion_folder_path):
            emotion_label = emotion_folder_name # El nombre de la carpeta es la etiqueta de la emoción
            print(f"  Cargando y pre-procesando imágenes para la emoción: {emotion_label}")

            for filename in os.listdir(emotion_folder_path):
                if filename.lower().endswith(image_extensions):
                    img_path = os.path.join(emotion_folder_path, filename)
                    try:
                        img = cv2.imread(img_path) # Cargar la imagen
                        if img is not None:
                            # Convertir a escala de grises
                            gray_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

                            # Redimensionar la imagen
                            resized_img = cv2.resize(gray_img, target_dim, interpolation=cv2.INTER_AREA)

                            processed_images.append(resized_img)
                            image_filenames.append(os.path.join(emotion_folder_name, filename)) # Guardar la ruta relativa
                            image_labels.append(emotion_label)
                            # Opcional: mostrar un progreso
                            # print(f"    Cargada y procesada: {os.path.join(emotion_folder_name, filename)}")
                        else:
                            print(f"    Advertencia: No se pudo cargar la imagen '{filename}' en '{emotion_folder_name}'. Posiblemente corrupta o formato no soportado.")
                    except Exception as e:
                        print(f"    Error al cargar o procesar la imagen '{filename}' en '{emotion_folder_name}': {e}")

    print(f"Total de imágenes cargadas y pre-procesadas: {len(processed_images)} de {image_base_folder_path}")

    # Las imágenes procesadas (en escala de grises y redimensionadas) están ahora en la lista 'processed_images',
    # sus nombres de archivo (relativos a la carpeta de emoción) en 'image_filenames',
    # y sus etiquetas de emoción en 'image_labels'.
    # Puedes acceder a ellas así:
    # for i, img_proc in enumerate(processed_images):
    #     print(f"Procesando imagen: {image_filenames[i]} (Emoción: {image_labels[i]}) con dimensiones {img_proc.shape}")

    # Ejemplo de visualización de las primeras 5 imágenes procesadas (opcional)
    # if processed_images:
    #     plt.figure(figsize=(15, 5))
    #     for i in range(min(5, len(processed_images))):
    #         plt.subplot(1, 5, i + 1)
    #         # Mostrar en escala de grises
    #         plt.imshow(processed_images[i], cmap='gray')
    #         plt.title(f"{image_filenames[i]} ({image_labels[i]})")
    #         plt.axis('off')
    #     plt.show()


Por favor, introduce la ruta completa a la carpeta principal que contiene las carpetas de emociones (ej: /content/drive/MyDrive/00.DataVxC/emotions_dataset/): /content/drive/MyDrive/00.DataVxC
Introduce el ancho deseado para las imágenes pre-procesadas (ej: 48): 48
Introduce el alto deseado para las imágenes pre-procesadas (ej: 48): 48
Las imágenes se pre-procesarán a dimensiones: (48, 48) en escala de grises.
Procesando imágenes de las subcarpetas en: /content/drive/MyDrive/00.DataVxC
  Cargando y pre-procesando imágenes para la emoción: Angry
  Cargando y pre-procesando imágenes para la emoción: Fear
  Cargando y pre-procesando imágenes para la emoción: Happy
  Cargando y pre-procesando imágenes para la emoción: Sad
  Cargando y pre-procesando imágenes para la emoción: Suprise
Total de imágenes cargadas y pre-procesadas: 59099 de /content/drive/MyDrive/00.DataVxC


Detección de rostros usando OpenCV DNN ResNet-10

# Task
Detect faces using OpenCV DNN ResNet-10 from the `processed_images` and store the detected face regions along with their corresponding `image_labels`.

## Detect faces

### Subtask:
Implement a function to detect faces in each pre-processed image using the chosen algorithm (OpenCV ResNet-10 or MediaPipe), extract the face regions, and store them along with their original labels.


**Reasoning**:
The subtask requires defining the paths to the pre-trained OpenCV DNN face detection model files (prototxt and caffemodel) and then loading this model using `cv2.dnn.readNetFromCaffe()`. This is the first logical step to prepare for face detection.



In [11]:
import cv2
import os
import numpy as np

# Define paths to the pre-trained model files
# Make sure these paths are correct and the files are accessible
# You might need to upload these files to your Google Drive or Colab environment

# Model files for OpenCV DNN face detector (ResNet-10 SSD)
prototxt_path = "/content/drive/MyDrive/models/deploy.prototxt.txt"  # Replace with your path
caffemodel_path = "/content/drive/MyDrive/models/res10_300x300_ssd_iter_140000.caffemodel" # Replace with your path

# Check if model files exist
if not os.path.exists(prototxt_path):
    print(f"Error: Prototxt file not found at {prototxt_path}")
    print("Please ensure the path is correct and the file exists.")
if not os.path.exists(caffemodel_path):
    print(f"Error: Caffe model file not found at {caffemodel_path}")
    print("Please ensure the path is correct and the file exists.")

# Load the pre-trained Caffe model
try:
    net = cv2.dnn.readNetFromCaffe(prototxt_path, caffemodel_path)
    print("OpenCV DNN face detection model loaded successfully.")
except Exception as e:
    print(f"Error loading Caffe model: {e}")
    print("Please check the file paths and ensure the model files are not corrupted.")

# Initialize lists to store detected face regions and their labels
face_regions = []
face_labels = []

print("Initialized 'face_regions' and 'face_labels' lists.")

# Define a confidence threshold for detections
confidence_threshold = 0.5
print(f"Using a confidence threshold of {confidence_threshold} for face detection.")

OpenCV DNN face detection model loaded successfully.
Initialized 'face_regions' and 'face_labels' lists.
Using a confidence threshold of 0.5 for face detection.


**Reasoning**:
The previous step failed to load the model due to file not found errors. However, the subtask requires implementing the face detection logic using the `net` object. To proceed with the subtask's instructions, I will now provide the code for iterating through the pre-processed images, performing face detection using the `net` (if it was successfully loaded), and storing the detected face regions and their corresponding labels. I will include a check to ensure `net` exists before attempting to use it.



In [9]:
if 'net' not in locals() or net is None:
    print("Error: Face detection model (net) was not loaded successfully in the previous step. Please ensure the prototxt and caffemodel files are correctly placed and accessible.")
else:
    print("Starting face detection process...")
    # Iterate through each pre-processed image and its label
    for i, img in enumerate(processed_images):
        label = image_labels[i]

        # Get image dimensions
        (h, w) = img.shape[:2]

        # Create a 4D blob from the image
        # ResNet-10 SSD expects 300x300 BGR images, even if input is grayscale
        # The mean subtraction values are standard for this model
        # Convert grayscale img to 3-channel for blobFromImage
        img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
        blob = cv2.dnn.blobFromImage(cv2.resize(img_rgb, (300, 300)), 1.0,
            (300, 300), (104.0, 177.0, 123.0))

        # Set the input and get detections
        net.setInput(blob)
        detections = net.forward()

        # Loop over the detections
        for j in range(0, detections.shape[2]):
            # Extract the confidence (probability) associated with the prediction
            confidence = detections[0, 0, j, 2]

            # Filter out weak detections by ensuring the confidence is greater than the minimum confidence
            if confidence > confidence_threshold:
                # Compute the (x, y)-coordinates of the bounding box for the object
                box = detections[0, 0, j, 3:7] * np.array([w, h, w, h])
                (startX, startY, endX, endY) = box.astype("int")

                # Ensure the bounding box coordinates fall within the dimensions of the frame
                startX = max(0, startX)
                startY = max(0, startY)
                endX = min(w, endX)
                endY = min(h, endY)

                # Extract the face region
                face = img[startY:endY, startX:endX]

                # Only consider non-empty face regions
                if face.shape[0] > 0 and face.shape[1] > 0:
                    face_regions.append(face)
                    face_labels.append(label)

    print(f"Finished face detection. Total faces detected: {len(face_regions)}")
    if len(face_regions) > 0:
        print(f"Example of a detected face region shape: {face_regions[0].shape}")


Error: Face detection model (net) was not loaded successfully in the previous step. Please ensure the prototxt and caffemodel files are correctly placed and accessible.
